# AI Music OS — ACE-Step V6

Free local models; no API key. Choose **T4 GPU**, then **Run all** in a fresh Colab session. Open the printed Gradio link. Each notebook uses its own pinned Python 3.11 environment, leaving Colab's Python 3.13 packages untouched. Models and WAV outputs remain on Drive. Keep the notebook session open; free Colab can disconnect it.

Upload this notebook directly to Colab, or replace its matching file in GitHub. No other repository files are required. Dependency and CPU import tests passed locally; Colab T4 inference is not certified.

**Setup revision V6:** does not use Colab system venv/ensurepip. Replace all three notebooks, not just the voice-clone file.


In [ ]:
MODULE = 'ace'
# Separate Python 3.11 environment; do not install AI packages into Colab's kernel.
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, time, re
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/AI_Music_OS')
CACHE = ROOT / 'cache' / 'huggingface'
OUT = ROOT / 'outputs' / {'ace': 'ace_step', 'kokoro': 'kokoro', 'f5': 'qwen_voice'}[MODULE]
VOICE_BANK = ROOT / 'voice_bank'
MODELS = ROOT / 'models' / 'ace_step'
for p in (CACHE, OUT, VOICE_BANK, MODELS): p.mkdir(parents=True, exist_ok=True)
ENV = os.environ.copy()
ENV['HF_HOME'] = str(CACHE)
ENV['HUGGINGFACE_HUB_CACHE'] = str(CACHE / 'hub')
ENV.pop('TRANSFORMERS_CACHE', None)
ENV['PYTHONUNBUFFERED'] = '1'
# Prevent Colab kernel paths from leaking into the isolated interpreter.
ENV.pop('PYTHONPATH', None)
ENV.pop('PYTHONHOME', None)
ENV['PYTHONNOUSERSITE'] = '1'
WORK = Path('/content/ai_music_v6') / MODULE
WORK.mkdir(parents=True, exist_ok=True)
LOG = OUT / 'install_v6.log'
def run_logged(cmd, cwd=None):
    with LOG.open('a') as log:
        result = subprocess.run(cmd, cwd=cwd, env=ENV, stdout=log, stderr=subprocess.STDOUT)
    if result.returncode:
        print(LOG.read_text(errors='replace')[-18000:])
        raise RuntimeError(f'Command failed ({result.returncode}). Complete output: {LOG}')
# Colab's system Python can fail at venv/ensurepip. Download the official uv
# wheel with existing pip and extract only its binary, without installing it
# into the kernel or running ensurepip. uv manages the separate AI interpreter.
import zipfile, shutil
UV_HOME = Path('/content/ai_music_v6/bootstrap')
UV_HOME.mkdir(parents=True, exist_ok=True)
UV = UV_HOME / 'uv'
if not UV.exists():
    WHEELS = UV_HOME / 'wheels'
    WHEELS.mkdir(exist_ok=True)
    run_logged([sys.executable, '-m', 'pip', 'download', '--no-cache-dir',
                '--no-deps', '--only-binary=:all:', '--index-url', 'https://pypi.org/simple',
                '--dest', str(WHEELS), 'uv==0.13.8'])
    wheels = sorted(WHEELS.glob('uv-0.13.8-*.whl'))
    if len(wheels) != 1:
        raise RuntimeError(f'Expected one uv wheel, found {len(wheels)}. See {LOG}')
    temporary = UV.with_suffix('.tmp')
    try:
        with zipfile.ZipFile(wheels[0]) as archive:
            binaries = [name for name in archive.namelist() if name.endswith('.data/scripts/uv')]
            if len(binaries) != 1: raise RuntimeError('Official uv wheel has no unique Linux binary.')
            with archive.open(binaries[0]) as src, temporary.open('wb') as dst:
                shutil.copyfileobj(src, dst)
        temporary.chmod(0o755)
        temporary.replace(UV)
    finally:
        temporary.unlink(missing_ok=True)
run_logged([str(UV), '--version'])
run_logged([str(UV), 'python', 'install', '3.11.18'])
VENV = WORK / 'venv'
PY = VENV / 'bin' / 'python'
if not PY.exists(): run_logged([str(UV), 'venv', '--python', '3.11.18', str(VENV)])
print('Colab kernel:', sys.version.split()[0], '| AI runtime: Python 3.11.18')
print('Outputs:', OUT, '| Installation log:', LOG)


In [ ]:
LOCK_CONTENT = 'absl-py==2.5.1\naccelerate==1.6.0\nace-step @ https://github.com/ace-step/ACE-Step/archive/1bee4c9f5b43e30995f8d4d33b3919197ce1bd68.zip\naiofiles==23.2.1\naiohappyeyeballs==2.7.1\naiohttp==3.14.5\naiosignal==1.4.0\nannotated-doc==0.0.6\nannotated-types==0.8.0\nanyio==4.16.0\nattrs==26.1.0\naudioread==3.1.0\nblis==1.3.4\ncatalogue==2.0.10\ncertifi==2026.10.23\ncffi==2.1.1\ncharset-normalizer==3.5.2\nclick==8.5.1\ncloudpathlib==0.26.0\ncloudpickle==3.1.2\nconfection==0.1.5\ncontourpy==1.3.3\ncutlet==0.5.2\ncycler==0.12.1\ncymem==2.0.16\ndatasets==3.4.1\ndecorator==5.3.1\ndiffusers==0.33.1\ndill==0.3.8\ndocopt==0.6.2\nfastapi==0.162.7\nffmpy==1.0.0\nfilelock==4.2.10\nfonttools==4.68.0\nfrozenlist==1.8.0\nfsspec==2024.12.0\nfugashi==1.5.2\ngradio==5.22.0\ngradio-client==1.8.0\ngroovy==0.1.2\ngrpcio==1.85.1\nh11==0.16.0\nhangul-romanize==0.1.0\nhttpcore==1.0.9\nhttpx==0.28.1\nhuggingface-hub==0.30.2\nidna==3.20\nimportlib-metadata==10.2.0\njaconv==0.5.0\njinja2==3.1.6\njoblib==1.6.0\nkiwisolver==1.5.1\nlangcodes==3.5.1\nlazy-importer==0.1.0\nlazy-loader==0.6\nlibrosa==0.11.0\nlightning-utilities==0.15.3\nllvmlite==0.50.0\nloguru==0.7.3\nmarkdown==3.11.3\nmarkdown-it-py==4.3.0\nmarkupsafe==3.0.4\nmatplotlib==3.10.1\nmdurl==0.1.2\nmojimoji==0.0.13\nmpmath==1.3.0\nmsgpack==1.2.4\nmultidict==7.2.1\nmultiprocess==0.70.16\nmurmurhash==1.0.16\nnarwhals==2.28.0\nnetworkx==3.6.1\nnum2words==0.5.14\nnumba==0.68.1\nnumpy==2.2.6\nnvidia-cublas-cu12==12.4.5.8\nnvidia-cuda-cupti-cu12==12.4.127\nnvidia-cuda-nvrtc-cu12==12.4.127\nnvidia-cuda-runtime-cu12==12.4.127\nnvidia-cudnn-cu12==9.1.0.70\nnvidia-cufft-cu12==11.2.1.3\nnvidia-curand-cu12==10.3.5.147\nnvidia-cusolver-cu12==11.6.1.9\nnvidia-cusparse-cu12==12.3.1.170\nnvidia-cusparselt-cu12==0.6.2\nnvidia-nccl-cu12==2.21.5\nnvidia-nvjitlink-cu12==12.4.127\nnvidia-nvtx-cu12==12.4.127\nopentelemetry-api==1.45.2\norjson==3.13.0\npackaging==26.3\npandas==2.3.3\npeft==0.15.2\npillow==11.3.0\npip==25.1.1\nplatformdirs==4.14.5\npooch==1.9.0\npreshed==3.0.15\npropcache==0.5.4\nprotobuf==7.36.2\npsutil==7.2.2\npy3langid==0.3.0\npyarrow==26.0.1\npycparser==3.12\npydantic==2.10.6\npydantic-core==2.27.2\npydub==0.25.1\npygments==2.21.0\npyparsing==3.3.3\npypinyin==0.53.0\npython-dateutil==4.0.0\npython-multipart==0.0.33\npytorch-lightning==2.5.1\npytz==2026.6\npyyaml==6.0.3\nregex==2026.11.9\nrequests==2.34.2\nrich==15.0.0\nruff==0.17.5\nsafehttpx==0.1.7\nsafetensors==0.8.0\nscikit-learn==1.9.1\nscipy==1.15.3\nsemantic-version==2.10.0\nsetuptools==84.2.1\nshellingham==1.5.4\nsmart-open==7.7.1\nsoundfile==0.13.1\nsoxr==1.1.0\nspacy==3.8.4\nspacy-legacy==3.0.12\nspacy-loggers==1.0.5\nsrsly==2.5.6\nstarlette==0.52.1\nsympy==1.13.1\ntensorboard==2.21.0\ntensorboard-data-server==0.7.2\ntensorboardx==2.6.5\nthinc==8.3.11\nthreadpoolctl==3.8.0\ntokenizers==0.21.4\ntomlkit==0.13.3\ntorch==2.6.0+cu124\ntorchaudio==2.6.0+cu124\ntorchmetrics==1.9.0\ntorchvision==0.21.0+cu124\ntqdm==4.70.1\ntransformers==4.50.0\ntriton==3.2.0\ntyper==0.29.1\ntyper-slim==0.24.0\ntyping-extensions==4.16.0\ntyping-inspection==0.4.5\ntzdata==2026.6\nunidic-lite==1.0.8\nurllib3==2.8.0\nuvicorn==0.54.0\nwasabi==1.1.3\nweasel==0.4.3\nwebsockets==15.0.1\nwerkzeug==3.1.9\nwrapt==2.5.2\nxxhash==4.1.0\nyarl==1.25.1\nzipp==4.1.2\n'
CHECK_IMPORTS = 'import torch, torchaudio, torchvision; from acestep.gui import main; print("ACE-Step imports: OK")'
# Full dependency resolution from a pinned lock, including addict and phonemizers.
LOCK = WORK / 'requirements.lock'
LOCK.write_text(LOCK_CONTENT)
run_logged([str(UV), 'pip', 'sync', '--python', str(PY), '--no-cache',
            '--index', 'https://download.pytorch.org/whl/cu124',
            '--index', 'https://pypi.org/simple', '--index-strategy', 'unsafe-best-match', str(LOCK)])
run_logged([str(UV), 'pip', 'check', '--python', str(PY)])
run_logged([str(PY), '-c', CHECK_IMPORTS])
print('Complete installation + imports: OK')


In [ ]:
# Verify the actual AI runtime, not Colab's preinstalled torch.
run_logged([str(PY), '-c', "import torch; print('Torch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'); assert torch.cuda.is_available(), 'Select T4 GPU in Runtime > Change runtime type, then rerun.'"])
print('T4/CUDA check: OK')


In [ ]:
# Confirm the official CLI before launching with T4 memory settings.
EXE = VENV / 'bin' / 'acestep'
run_logged([str(EXE), '--help'])
COMMAND = [str(EXE), '--checkpoint_path', str(MODELS), '--port', '7865',
           '--share', 'true', '--bf16', 'false', '--torch_compile', 'false',
           '--cpu_offload', 'true', '--overlapped_decode', 'true']
# Official ACE-Step writes ./outputs; working directory is on Drive.


In [ ]:
# Launch in the isolated environment; rerunning safely replaces the previous process.
if 'proc' in globals() and proc.poll() is None:
    proc.terminate()
    try: proc.wait(timeout=20)
    except subprocess.TimeoutExpired: proc.kill(); proc.wait()
LAUNCH_LOG = OUT / 'launch_v6.log'
with LAUNCH_LOG.open('w') as log:
    proc = subprocess.Popen(COMMAND, cwd=str(OUT), env=ENV, stdout=log, stderr=subprocess.STDOUT)
url = None
for _ in range(900):
    time.sleep(2)
    text = LAUNCH_LOG.read_text(errors='replace')
    if proc.poll() is not None:
        print(text[-18000:])
        raise RuntimeError(f'Application exited. Full log: {LAUNCH_LOG}')
    match = re.search(r'https://[a-zA-Z0-9-]+\.gradio\.live', text)
    if match: url = match.group(0); break
if url is None:
    print(LAUNCH_LOG.read_text(errors='replace')[-6000:])
    raise TimeoutError(f'No share URL after 30 minutes; inspect {LAUNCH_LOG}. Rerun this cell to retry.')
print('Open Music OS:', url)
print('Keep this Colab session open. Logs:', LAUNCH_LOG)


In [ ]:
# Inspect the application log at any time.
print(LAUNCH_LOG.read_text(errors='replace')[-15000:])
